[OrthoFinder](https://github.com/davidemms/OrthoFinder).
```
conda install ipykernel orthofinder requests tqdm
```


# Dependencies

In [1]:
from pathlib import Path
import subprocess
import shutil
import sys
import requests
import tqdm

# --- Import Python Utilities ---
# -------------------------------
relative_target_path = Path("utils") / "python_utils"
project_root = Path.cwd()
found_root = None
while True:
    if (project_root / relative_target_path).is_dir():
        found_root = project_root
        break # Found it!
    # Stop if we reach the filesystem root
    if project_root == project_root.parent:
        raise FileNotFoundError(
            f"Could not find the directory structure '{relative_target_path}'"
        )
    # Go one level up for the next iteration
    project_root = project_root.parent
# Add the found project root to sys.path if it's not already there
if found_root:
    path_str = str(found_root)
    if path_str not in sys.path:
        sys.path.append(path_str)
        print(f"Added '{path_str}' to sys.path")

from utils.python_utils import (
    # Directory Paths
    workflow_dir,
    # Functions
    execute_command,
    compression_utility
)

Added '/home/arnek/Tomoseq_workflow' to sys.path


# Download the proteome sequences
The selection of species has to provide good coverage between the phylogenetic branches of the target species *M. musculus* and *A. cahirinus*. Most putative proteomes were downloaded from [Ensembl Rapid Release](https://ftp.ensembl.org/pub/rapid-release/species/) 

## Define download URLs

In [ ]:
# --- Target species and their subfamilies ---
# --------------------------------------------
# --- Mus ---
# Mus musculus (GRCm39)
mus_mus = 'https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/pep/Mus_musculus.GRCm39.pep.all.fa.gz'
# Mus caroli (CAROLI_EIJ_v1.1)
mus_car = 'https://ftp.ensembl.org/pub/release-113/fasta/mus_caroli/pep/Mus_caroli.CAROLI_EIJ_v1.1.pep.all.fa.gz'
# Mus pahari (PAHARI_EIJ_v1.1)
mus_pah = 'https://ftp.ensembl.org/pub/release-113/fasta/mus_pahari/pep/Mus_pahari.PAHARI_EIJ_v1.1.pep.all.fa.gz'
# Mus minutoides (GCA_902729485.2-2023_03)
mus_min = 'https://ftp.ensembl.org/pub/rapid-release/species/Mus_minutoides/GCA_902729485.2/ensembl/geneset/2023_03/Mus_minutoides-GCA_902729485.2-2023_03-pep.fa.gz'

# --- Acomys ---
# Acomys cahirinus (GCA_029890205.1-2023_11)
aco_cah = 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/geneset/2023_11/Acomys_cahirinus-GCA_029890205.1-2023_11-pep.fa.gz'
# Acomys russatus (GCA_903995435.1-2023_10)
aco_rus = 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_russatus/GCA_903995435.1/ensembl/geneset/2023_10/Acomys_russatus-GCA_903995435.1-2023_10-pep.fa.gz'
# Acomys dimidiatus (GCA_907164435.1-2022_07)
aco_dim = 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_dimidiatus/GCA_907164435.1/ensembl/geneset/2022_07/Acomys_dimidiatus-GCA_907164435.1-2022_07-pep.fa.gz'

# --- Between-group species ---
# -----------------------------
# Arvicanthis niloticus (Arvicanthini) (GCA_011762505.1-2020_06)
arvi_nil = 'https://ftp.ensembl.org/pub/rapid-release/species/Arvicanthis_niloticus/GCA_011762505.1/ensembl/geneset/2020_06/Arvicanthis_niloticus-GCA_011762505.1-2020_06-pep.fa.gz'
# Meriones unguiculatus (Gerbillinae) (MunDraft-v1.0)
meri_ung = 'https://ftp.ensembl.org/pub/rapid-release/species/Meriones_unguiculatus/GCA_030254825.1/ensembl/geneset/2023_11/Meriones_unguiculatus-GCA_030254825.1-2023_11-pep.fa.gz'
# Apodemus agaricus (Apodemyini) (GCA_964023405.1-2024_05)
apo_agar = 'https://ftp.ensembl.org/pub/rapid-release/species/Apodemus_agrarius/GCA_964023405.1/ensembl/geneset/2024_05/Apodemus_agrarius-GCA_964023405.1-2024_05-pep.fa.gz'
# Rattus norvegicus (Rattini) (mRatBN7.2);
rat_norv = 'https://ftp.ensembl.org/pub/release-113/fasta/rattus_norvegicus/pep/Rattus_norvegicus.mRatBN7.2.pep.all.fa.gz'
# Mastomys coucha (Praomyini) (GCA_008632895.1-2023_10)
masto_cou = 'https://ftp.ensembl.org/pub/rapid-release/species/Mastomys_coucha/GCA_008632895.1/ensembl/geneset/2023_10/Mastomys_coucha-GCA_008632895.1-2023_10-pep.fa.gz'
# Lophiomys imhausi (Lophiomyinae) (GCA_907164525.1-2024_05)
lophi_imha = 'https://ftp.ensembl.org/pub/rapid-release/species/Lophiomys_imhausi/GCA_907164525.1/ensembl/geneset/2024_05/Lophiomys_imhausi-GCA_907164525.1-2024_05-pep.fa.gz'

# --- Out-group species ---
# -------------------------
# Orcytolagus cuniculus (Rabbit) (OryCun2.0)
orcy_cuni = 'https://ftp.ensembl.org/pub/release-113/fasta/oryctolagus_cuniculus/pep/Oryctolagus_cuniculus.OryCun2.0.pep.all.fa.gz'
# Cavia porcellus (Guinea pig) (Cavpor3.0)
cavi_porce = 'https://ftp.ensembl.org/pub/release-113/fasta/cavia_porcellus/pep/Cavia_porcellus.Cavpor3.0.pep.all.fa.gz'
# Chinchilla_lanigera (ChiLan1.0)
chi_lan = 'https://ftp.ensembl.org/pub/release-113/fasta/chinchilla_lanigera/pep/Chinchilla_lanigera.ChiLan1.0.pep.all.fa.gz'

species_url_list = [
    # Mus
    mus_mus,
    mus_car,
    mus_pah,
    mus_min,
    # Acomys
    aco_cah,
    aco_rus,
    aco_dim,
    # between-group
    arvi_nil,
    meri_ung,
    apo_agar,
    rat_norv,
    masto_cou,
    lophi_imha,
    # out-group
    orcy_cuni,
    cavi_porce,
    chi_lan
]


## Download

In [ ]:
orthofinder_dir = workflow_dir / 'orthofinder'
orthofinder_proteomes_dir = orthofinder_dir / 'proteomes'
orthofinder_proteomes_dir.mkdir(exist_ok=True, parents=True)

def download_web_file(
        url: str,
        output_file: Path):
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
            # Get file size from headers (in bytes)
            total_size = int(r.headers.get('content-length', 0))
            with open(output_file, 'wb') as f:
                with tqdm.tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"Downloading {output_file.name}",
                    bar_format="{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
        if not output_file.is_file() or not output_file.stat().st_size > 0:
            print(f"Error: Failed to download {url} to {output_file}")
            return False
        print(f"Successfully downloaded {output_file}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")
        return False

for species_url in species_url_list:
    output_file_name = species_url.split('/')[-1]
    download_success = download_web_file(
        url=species_url,
        output_file= Path(orthofinder_proteomes_dir) / output_file_name)
if not download_success:
    print(f"Failed to download {species_url}")

# Proteome pre-processing
It is recommended that only the longest translated sequence per gene is retained: [OrthoFinder Tutorial](https://davidemms.github.io/orthofinder_tutorials/running-an-example-orthofinder-analysis.html). 


In [ ]:
# Script path 
orthofinder_dir = workflow_dir / 'orthofinder'
primary_filter_script = Path(orthofinder_dir) / 'primary_transcript.py'

# Fetch the peptide fastas
orthofinder_proteomes_dir = orthofinder_dir / 'proteomes'
pep_fa_list = orthofinder_proteomes_dir.glob('*.fa*')
pep_fa_list = [path for path in pep_fa_list if path.is_file()]
if not pep_fa_list:
    raise FileNotFoundError(f"No peptide fasta files in {orthofinder_proteomes_dir}")
print(f"Found {len(pep_fa_list)} peptide fasta files in {orthofinder_proteomes_dir.name}")

# Iteratively process the peptide fastas
for pep_fa in pep_fa_list:
    # Track variables for cleanup
    pep_fa_decompressed = None
    try:
        print(f"\nprocessing {pep_fa.name}")
        print("-------------------------------")
        
        # --- Decompress the peptide fasta if necessary ---
        if pep_fa.suffix == '.gz':
            pep_fa_decompressed = pep_fa.with_suffix('')
            decompression_success = compression_utility(
                input_path=pep_fa,
                output_path=pep_fa_decompressed,
                compress=False
            )
            if not decompression_success \
                or not pep_fa_decompressed.is_file() \
                or not pep_fa_decompressed.stat().st_size > 0:
                    raise RuntimeError(f"Failed to decompress {pep_fa} to {pep_fa_decompressed}")
            pep_fa_to_process = pep_fa_decompressed
        else:
            pep_fa_to_process = pep_fa

        # --- Run the primary filter script ---
        primary_filter_cmd = [
            'python', primary_filter_script,
            pep_fa_to_process]
        exit_status = execute_command(primary_filter_cmd)
        if exit_status != 0:
            raise RuntimeError(f"Failed to execute {primary_filter_cmd}")
    finally:
        if pep_fa_decompressed and pep_fa_decompressed.is_file():
            pep_fa_decompressed.unlink()
    

Found 13 peptide fasta files in proteomes
>> processing Meriones_unguiculatus-GCA_030254825.1-2023_11-pep.fa.gz
Decompressing gzip file 'Meriones_unguiculatus-GCA_030254825.1-2023_11-pep.fa.gz' to 'Meriones_unguiculatus-GCA_030254825.1-2023_11-pep.fa'
Gzip file decompression successful.

Looking for "gene=" of "gene:" to identify isoforms of same gene
Found 39338 accessions, 19631 genes, 0 unidentified transcripts
Wrote 19631 genes
/home/arnek/Tomoseq_workflow/orthofinder/proteomes/primary_transcripts/Meriones_unguiculatus-GCA_030254825.1-2023_11-pep.fa
>> processing Acomys_dimidiatus-GCA_907164435.1-2022_07-pep.fa.gz
Decompressing gzip file 'Acomys_dimidiatus-GCA_907164435.1-2022_07-pep.fa.gz' to 'Acomys_dimidiatus-GCA_907164435.1-2022_07-pep.fa'
Gzip file decompression successful.

Looking for "gene=" of "gene:" to identify isoforms of same gene
Found 40980 accessions, 22172 genes, 0 unidentified transcripts
Wrote 22172 genes
/home/arnek/Tomoseq_workflow/orthofinder/proteomes/primary

# Run OrthoFinder

In [4]:
orthofinder_dir = workflow_dir / 'orthofinder'
orthofinder_proteomes_dir = orthofinder_dir / 'proteomes'
primary_proteomes_dir = orthofinder_proteomes_dir / 'primary_transcripts'
if not primary_proteomes_dir.is_dir():
    raise FileNotFoundError(f"Primary proteomes directory not found: {primary_proteomes_dir}")

output_dir = Path(orthofinder_dir) / 'orthofinder_results'
orthofinder_command = [
    'orthofinder', 
    '-t', '4', # Number of parallel sequence search threads [Default = 6]
    '-a', '2', # Number of parallel analysis threads
    '-o', str(output_dir),
    '-f', str(primary_proteomes_dir)
]
exit_status = execute_command(orthofinder_command)
if exit_status != 0:
    raise RuntimeError(f"Failed to execute command {orthofinder_command}")


OrthoFinder version 3.0.1b1 Copyright (C) 2014 David Emms

2025-04-24 13:23:28 : Starting OrthoFinder 3.0.1b1
4 thread(s) for highly parallel tasks (BLAST searches etc.)
2 thread(s) for OrthoFinder algorithm

Results directory: /home/arnek/Tomoseq_workflow/orthofinder/orthofinder_results/Results_Apr24/

Checking required programs are installed
----------------------------------------
Running with the recommended MSA tree inference by default. To revert to legacy method use '-M dendroblast'.

Test can run "mcl -h" - ok
Test can run "mafft" - ok
Test can run "fasttree" - ok

Dividing up work for BLAST for parallel processing
--------------------------------------------------
2025-04-24 13:23:29 : Creating diamond database 1 of 13
2025-04-24 13:23:29 : Creating diamond database 2 of 13
2025-04-24 13:23:29 : Creating diamond database 3 of 13
2025-04-24 13:23:30 : Creating diamond database 4 of 13
2025-04-24 13:23:30 : Creating diamond database 5 of 13
2025-04-24 13:23:30 : Creating diamon